In [1]:
%load_ext autoreload
%autoreload 2

In [6]:
from ariel_pred.dataset import DataLoaderAndCalibrator
from ariel_pred.preprocessing import SGSmoothing
from ariel_pred.plots import plot_white_curve
from ariel_pred.transit import FunctionFittingBasedPhaseDetector
from ariel_pred.models import TransitMultiplicationFactorFinder, SValuesCNN, SValuesCNNTrainer
from ariel_pred.features import WavelengthsGroupsMultiplierFinder
from ariel_pred.metrics import score, gll
from ariel_pred.sigma import SpectrumVariationScaler
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import random
from scipy.optimize import minimize
from tqdm.auto import tqdm
import pandas as pd
import torch
from torch import nn
import scipy

In [3]:
data_loader = DataLoaderAndCalibrator(
    data_path=Path("../data/raw"),
    output_path=Path("../data/calibrated/full"),
    force_recalibration=False,
    cut_airs_channels=True,
    binning=4,
    n_jobs=4
)
train_data, train_labels = data_loader.load_all_train_data()
train_data.shape, train_labels.shape

Loading calibrated train data...


((1100, 1406, 283), (1100, 283))

In [4]:
features_extractor = WavelengthsGroupsMultiplierFinder()

features = features_extractor.extract_features(
    train_data,
)

features.shape

100%|██████████| 1100/1100 [03:22<00:00,  5.43it/s]


(1100, 283)

In [22]:
spectrum = 0.95 * features
sigma_calculator = SpectrumVariationScaler(mean_sigma=0.00063, num_channels=283)
predicted_sigma = sigma_calculator.get_sigma(spectrum)
predicted_sigma.shape

(1100, 283)

In [23]:
gll(np.concatenate([spectrum, predicted_sigma], axis=1), train_labels)

0.3852957442115603

In [21]:
def cost_function(params):
    multiplier, mean_sigma = params
    spectrum = multiplier * features
    sigma_calculator = SpectrumVariationScaler(mean_sigma=mean_sigma, num_channels=283)
    predicted_sigma = sigma_calculator.get_sigma(spectrum)
    return -gll(np.concatenate([spectrum, predicted_sigma], axis=1), train_labels)

initial_guess = [0.945, 0.00068]
result = minimize(cost_function, initial_guess, method='Nelder-Mead')
optimal_multiplier, optimal_mean_sigma = result.x
optimal_multiplier, optimal_mean_sigma, -result.fun

(np.float64(0.9521266344487669),
 np.float64(0.0006327501231431961),
 np.float64(0.38563380170888445))